In [ ]:
from datetime import datetime

from vnpy_portfoliostrategy import BacktestingEngine
from vnpy.trader.constant import Interval
from vnpy.trader.optimize import OptimizationSetting

from vnpy_portfoliostrategy.strategies.pair_trading_strategy import PairTradingStrategy

In [ ]:
engine = BacktestingEngine()
engine.set_parameters(
    vt_symbols=["600031.SSE", "600036.SSE"],   # 你数据库里有日线数据的2个标的
    interval=Interval.DAILY,                   # 必须和数据库K线周期一致
    start=datetime(2023, 1, 1),
    end=datetime(2025, 12, 31),                # 时间拉长，增加触发机会
    rates={
        "600031.SSE": 0.0003,
        "600036.SSE": 0.0003
    },
    slippages={
        "600031.SSE": 0.01,
        "600036.SSE": 0.01
    },
    sizes={
        "600031.SSE": 1,
        "600036.SSE": 1
    },
    priceticks={
        "600031.SSE": 0.01,
        "600036.SSE": 0.01
    },
    capital=1_000_000,
)

# 先用“容易触发”的参数
setting = {
    "boll_window": 10,   # 先小窗口
    "boll_dev": 0.8,     # 先小倍数，触发更频繁
}
engine.add_strategy(PairTradingStrategy, setting)

In [ ]:
engine.load_data()
engine.run_backtesting()
df = engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

In [ ]:
setting = OptimizationSetting()
setting.set_target("sharpe_ratio")
setting.add_parameter("boll_window", 10, 30, 1)
setting.add_parameter("boll_dev", 1, 3, 1)

engine.run_ga_optimization(setting)

In [ ]:
engine.run_bf_optimization(setting)

In [ ]:
from vnpy.trader.database import get_database

target = {"600031.SSE", "600036.SSE"}
db = get_database()

print("=== bar_overview ===")
for o in db.get_bar_overview():
    vt = f"{o.symbol}.{o.exchange.value}"
    if vt in target:
        print(vt, o.interval.value, o.count, o.start, o.end)

In [ ]:
from vnpy.trader.database import get_database

db = get_database()
all_overviews = db.get_bar_overview()

print("总条目数:", len(all_overviews))
for o in all_overviews[:30]:
    print(f"{o.symbol}.{o.exchange.value}", o.interval.value, o.count, o.start, o.end)

In [ ]:
from datetime import datetime
from vnpy.trader.utility import extract_vt_symbol
from vnpy.trader.constant import Interval

db = get_database()
for vt in ["600031.SSE", "600036.SSE"]:
    symbol, exchange = extract_vt_symbol(vt)
    bars = db.load_bar_data(
        symbol=symbol,
        exchange=exchange,
        interval=Interval.DAILY,
        start=datetime(2025, 1, 1),
        end=datetime(2025, 4, 30)
    )
    print(vt, "d条数:", len(bars))
